# GTR Upfront Costs Innovation Analysis

## 📋 Overview

**Purpose**: Analyzes upfront cost innovation research projects from UK Research & Innovation's Gateway to Research (GtR) using LLM-validated filtering to identify technologies for reducing heat pump deployment costs.

**Key Features**:
- LLM validation of research project relevance to upfront cost reduction
- Time series analysis of research funding and project counts
- Comprehensive chart generation with both PNG and HTML outputs
- Automated Google Sheets export with chart data
- Organized directory structure for professional data management

**Expected Results**: 
- Upfront Cost Innovation Projects: ~50-80 research projects
- Focus on UKRI research funding patterns for cost reduction technology
- Analysis of academic/research trends rather than commercial investment

---

In [1]:
from discovery_utils.getters import gtr
from discovery_utils.utils import search
from discovery_utils.utils import analysis
from discovery_utils.utils.llm.batch_check import LLMProcessor, generate_relevance_check_system_message
from discovery_mission_radar import PROJECT_DIR

/Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-30 17:09:02,358 - datasets - INFO - PyTorch version 2.4.1 available.


## ⚙️ Technical Requirements

**Prerequisites**:
- `discovery_utils` and `discovery_mission_radar` packages installed
- AWS credentials configured for GTR S3 access
- OpenAI API key for LLM relevance checking
- Google Sheets API service account credentials

**Key Configuration Parameters**:
- `SCORE_THRESHOLD = 0.3` - Minimum relevance score for search results
- `config = "upfront_costs"` - Configuration file name
- Chart outputs: Both PNG (reports) and HTML (interactive) formats

**Required Files**:
- `config_upfront_costs.yaml` - Defines search terms for upfront cost innovation projects
- Vector database in `tmp/vector_db/` for semantic search

**Expected Runtime**:
- Quick run (existing data): 1-2 minutes
- Full run with LLM processing: 8-12 minutes  
- Memory usage: ~1GB peak during processing

---

In [2]:
OUTPUT_DIR = PROJECT_DIR / 'data/upfront_costs_innovation/gtr/'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DB_DIR = PROJECT_DIR / 'tmp/vector_db/'
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)
SCORE_THRESHOLD = 0.3

# Single config approach - simplified directory structure
primary_config = "upfront_costs"

# Create simplified directory structure directly under gtr/
raw_dir = OUTPUT_DIR / "raw"
filtered_dir = OUTPUT_DIR / "filtered"
produce_stats_dir = OUTPUT_DIR / "produce_stats"
charts_dir = produce_stats_dir / "charts"
chart_csvs_dir = produce_stats_dir / "chart_csvs"

# Create all directories
for directory in [raw_dir, filtered_dir, produce_stats_dir, charts_dir, chart_csvs_dir]: 
    directory.mkdir(exist_ok=True)

## 🚀 Usage Instructions

**Quick Start**: Run all cells sequentially. If filtered data exists, LLM processing will be skipped automatically.

**Key Parameters to Modify**:
```python
# Adjust filtering sensitivity
SCORE_THRESHOLD = 0.2  # Lower = more results, 0.4 = more selective

# Limit Google Sheets upload
SHEET_TO_UPDATE = None  # Set to specific sheet name to update only that sheet
```

**Output Structure** (Simplified):
- **Raw/Filtered**: `data/upfront_costs_innovation/gtr/raw/` and `filtered/`
- **Analysis**: `data/upfront_costs_innovation/gtr/produce_stats/` - charts, CSVs
- **Charts**: `data/upfront_costs_innovation/gtr/charts/` - visualization files
- **Chart Data**: `data/upfront_costs_innovation/gtr/chart_csvs/` - data for Google Sheets
- **Google Sheets**: Automatically uploaded with formatting applied

**Before Running**:
- Ensure `config_upfront_costs.yaml` exists in notebook directory
- Update Google Sheet ID in upload sections if needed
- Verify sufficient disk space (~50MB for outputs)

---

In [3]:
GTR = gtr.GtrGetter(vector_db_path=VECTOR_DB_DIR)

# Use primary config approach for consistency with other notebooks
config = primary_config
config_file = f'config_{config}.yaml'

print(f"Using configuration: {config}")
print(f"Config file: {config_file}")
print(f"Output directory: {OUTPUT_DIR}")

2025-07-30 17:19:10,357 - discovery_utils.getters.gtr - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-07-30 17:19:10,489 - discovery_utils.getters.gtr - INFO - Latest version found: GtR_20250626
Using configuration: upfront_costs
Config file: config_upfront_costs.yaml
Output directory: /Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/data/upfront_costs_innovation/gtr


## Data Search and LLM Filtering

In [4]:
# Check if processed data already exists
import pandas as pd

upfront_csv_path = OUTPUT_DIR / "filtered" / f"relevant_{primary_config}_llm_filtered.csv"

if upfront_csv_path.exists():
    print(f"✓ Found existing {primary_config} filtered data. Loading from file...")
    df_filtered = pd.read_csv(upfront_csv_path)
    print(f"Loaded {len(df_filtered)} {primary_config} research projects")
else:
    print(f"⚠️  No existing {primary_config} data found. Running full filtering process...")
    
    # Step 1: run keyword+vector search
    searcher = search.SearchDataset(GTR, GTR.projects_enriched, config_file)
    df = searcher.do_search()
    relevant = df.query(f"_score_avg > {SCORE_THRESHOLD}")
    csv_path_temp = OUTPUT_DIR / "raw" / f"relevant_{primary_config}.csv"
    jsonl_path = OUTPUT_DIR  / "raw" / f"llm_check_MS_{primary_config}.jsonl"
    relevant.to_csv(csv_path_temp, index=False)
    
    print(f"Found {len(relevant)} relevant projects above threshold {SCORE_THRESHOLD}")
    
    # Step 2: LLM-based relevance check
    sys_msg = generate_relevance_check_system_message(config_file)
    processor = LLMProcessor(
        output_path=str(jsonl_path),
        system_message=sys_msg,
        session_name='mission_studio',
        output_fields=[{'name':'is_relevant','type':'str','description':'yes or no'}]
    )
    await processor.run(dict(zip(relevant['id'], relevant['abstractText'])), batch_size=15, sleep_time=0.5)
    
    # Step 3: Merge and filter results
    df_csv = pd.read_csv(csv_path_temp)
    df_jsonl = pd.read_json(jsonl_path, lines=True)
    merged = df_csv.merge(df_jsonl[['id','is_relevant']], on='id', how='left')
    filtered = merged[merged['is_relevant'].str.lower()=='yes']
    rejected = merged[merged['is_relevant'].str.lower()=='no']
    
    # Save to organized directory structure
    filtered.to_csv(OUTPUT_DIR / "filtered" / f"relevant_{primary_config}_llm_filtered.csv", index=False)
    rejected.to_csv(OUTPUT_DIR / "filtered" / f"relevant_{primary_config}_rejected.csv", index=False)
    
    print(f"Filtered {len(filtered)} relevant entries out of {len(df_csv)} total.")
    print(f"Filtered {len(rejected)} rejected entries out of {len(df_csv)} total.")
    
    df_filtered = filtered.copy()

print(f"\n🎯 Data Summary:")
print(f"Upfront Cost Innovation: {len(df_filtered)} research projects")

⚠️  No existing upfront_costs data found. Running full filtering process...
2025-07-30 17:19:13,272 - discovery_utils.getters.gtr - INFO - Downloading parquet file: data/GtR/GtR_20250626/projects.parquet
2025-07-30 17:19:24,495 - discovery_utils.getters.gtr - INFO - Successfully downloaded and read parquet file: data/GtR/GtR_20250626/projects.parquet
2025-07-30 17:19:26,144 - discovery_utils.getters.gtr - INFO - Downloading parquet file: data/GtR/GtR_20250626/funds.parquet
2025-07-30 17:19:27,863 - discovery_utils.getters.gtr - INFO - Successfully downloaded and read parquet file: data/GtR/GtR_20250626/funds.parquet
2025-07-30 17:19:28,633 - root - INFO - Folder /Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/tmp/vector_db/gtr-lancedb/ already exists. Set overwrite=True to download again.
2025-07-30 17:19:28,636 - root - INFO - Connected with database gtr-lancedb. Available tables: ['project_embeddings']
2025-07-30 17:19:28,707 - root - ERROR - Error 

Batches: 100%|██████████| 1/1 [00:00<00:00, 102.03it/s]


2025-07-30 17:19:36,991 - discovery_utils - INFO - Vector search found 15863 matches
Found 366 relevant projects above threshold 0.3
2025-07-30 17:19:37,305 - root - INFO - Using OpenAI
2025-07-30 17:19:37,491 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client
2025-07-30 17:19:37,526 - root - INFO - Processing batch 1/25
2025-07-30 17:19:40,626 - root - INFO - Processing batch 2/25
2025-07-30 17:19:42,934 - root - INFO - Processing batch 3/25
2025-07-30 17:19:44,811 - root - INFO - Processing batch 4/25
2025-07-30 17:19:47,429 - root - INFO - Processing batch 5/25
2025-07-30 17:19:50,128 - root - INFO - Processing batch 6/25
2025-07-30 17:20:00,346 - openai._base_client - INFO - Retrying request to /chat/completions in 0.485094 seconds
2025-07-30 17:20:02,043 - root - INFO - Processing batch 7/25
2025-07-30 17:

In [5]:
# Analysis for GTR Upfront Cost Innovation projects
from pathlib import Path

# Create organized processing for the config
config_name = primary_config
print(f"=== Processing {config_name} ===")

# Create separate directory for this config
config_output_dir = OUTPUT_DIR
config_output_dir.mkdir(exist_ok=True)

# Load the filtered CSV from new directory structure
csv_path = config_output_dir / "filtered" / f"relevant_{config_name}_llm_filtered.csv"
df = pd.read_csv(csv_path, parse_dates=["start", "end"], dayfirst=False)

print(f"{len(df):,} projects loaded for {config_name}")
print(f"Date range: {df['start'].min().strftime('%Y-%m-%d')} to {df['end'].max().strftime('%Y-%m-%d')}")
print(f"Total funding: £{df['amount'].sum()/1_000_000:.1f}M")

display(df.head())

=== Processing upfront_costs ===
26 projects loaded for upfront_costs
Date range: 2009-10-01 to 2025-12-31
Total funding: £10.9M


,index,ext,id,outcomeid,href,created,updated,identifiers,title,status,...,end,funds_id,funds_category,currencyCode,amount,url,_score_keywords,_score_vectors,_score_avg,is_relevant
0,11939,NaN,D66D993B-09B1-4DC4-A3C4-46E417D94073,NaN,http://gtr.ukri.org/gtr/api/projects/D66D993B-...,1750273397000,NaN,{'identifier': array([{'value': 'EP/H008047/1'...,Gas-Fired Heat Pumps - Transforming the UK's D...,Closed,...,2010-09-30,C2F2BCD5-E7C9-4E12-80A8-0F86602A647D,INCOME_ACTUAL,GBP,97819,https://gtr.ukri.org/projects?ref=EP/H008047/1,0.0,0.990537,0.495269,yes
1,11193,NaN,53E0A653-8B5A-4CF7-ADAC-72C77431071B,NaN,http://gtr.ukri.org/gtr/api/projects/53E0A653-...,1750273397000,NaN,"{'identifier': array([{'value': '131509', 'typ...",Improved Domestic Air Source Heat Pump for bot...,Closed,...,2015-05-31,C3476A7C-D261-4C42-9417-E6B3B3736491,INCOME_ACTUAL,GBP,148012,https://gtr.ukri.org/projects?ref=131509,0.0,0.980026,0.490013,yes
2,15351,NaN,7F0EF388-E1E1-4ECB-B01F-177FD0C48EFD,NaN,http://gtr.ukri.org/gtr/api/projects/7F0EF388-...,1750273397000,NaN,"{'identifier': array([{'value': '10138296', 't...",Turbo Air Source Heat Pump Technology (Turbo-A...,Active,...,2025-10-31,9D1FF75B-A5FE-4111-B8F2-61227512A42A,INCOME_ACTUAL,GBP,299883,https://gtr.ukri.org/projects?ref=10138296,0.0,0.969745,0.484873,yes
3,246,NaN,6E7C2ACE-D6D8-40FC-AF0B-B03ADC41BA34,NaN,http://gtr.ukri.org/gtr/api/projects/6E7C2ACE-...,1750273397000,NaN,"{'identifier': array([{'value': '10040095', 't...",Measurement and Analysis of Compressor Materia...,Closed,...,2023-03-31,47138008-892D-4C6D-9DA0-8B9F0E159DDB,INCOME_ACTUAL,GBP,7501,https://gtr.ukri.org/projects?ref=10040095,0.0,0.959736,0.479868,yes
4,13315,NaN,169CA466-08DC-4BE4-84E7-69D7EC634A47,NaN,http://gtr.ukri.org/gtr/api/projects/169CA466-...,1750273397000,NaN,{'identifier': array([{'value': 'EP/V042033/1'...,Flexible Air Source Heat pump for domestic hea...,Closed,...,2023-12-31,E949AA0A-3751-417E-8860-2C1246D49BD4,INCOME_ACTUAL,GBP,1149351,https://gtr.ukri.org/projects?ref=EP/V042033/1,0.0,0.936235,0.468117,yes


In [6]:
# Generate time series data and save consolidated files
df["year"] = df["start"].dt.year
df["quarter"] = df["start"].dt.to_period("Q").astype(str)

# Yearly summary (before imputation)
ts_year_raw = (
    df.groupby("year", as_index=False)
      .agg(n_projects=("id", "count"),
           amount_millions=("amount", lambda s: s.sum()/1_000_000))
)

# Apply imputation to yearly data for better chart visualization
ts_year_for_imputation = ts_year_raw.copy()
ts_year_for_imputation['year_datetime'] = pd.to_datetime(ts_year_for_imputation['year'], format='%Y')

# Apply imputation using the datetime column
ts_year_imputed = analysis.impute_empty_periods(
    ts_year_for_imputation.drop('year', axis=1), 
    time_period_col='year_datetime', 
    period='Y',  # 'Y' for annual/yearly data
    min_year=2014, 
    max_year=2025
)

# Convert back to integer year for downstream functions
ts_year = ts_year_imputed.copy()
ts_year['year'] = ts_year['year_datetime'].dt.year
ts_year = ts_year.drop('year_datetime', axis=1)

# Quarterly summary (before imputation)
ts_quarter_raw = (
    df.groupby("quarter", as_index=False)
      .agg(n_projects=("id", "count"),
           amount_millions=("amount", lambda s: s.sum()/1_000_000))
)

# Apply imputation to quarterly data
ts_quarter_for_imputation = ts_quarter_raw.copy()
ts_quarter_for_imputation['quarter_datetime'] = pd.to_datetime(ts_quarter_for_imputation['quarter'])

# Apply quarterly imputation
ts_quarter_imputed = analysis.impute_empty_periods(
    ts_quarter_for_imputation.drop('quarter', axis=1),
    time_period_col='quarter_datetime',
    period='Q',  # 'Q' for quarterly data
    min_year=2014,
    max_year=2025
)

# Convert back to quarter string format
ts_quarter = ts_quarter_imputed.copy()
ts_quarter['quarter'] = ts_quarter['quarter_datetime'].dt.to_period('Q').astype(str)
ts_quarter = ts_quarter.drop('quarter_datetime', axis=1)

# Save consolidated CSV files to produce_stats directory
# These contain all the data for this particular config with theme column added
df.assign(theme=config_name).to_csv(produce_stats_dir / "all_projects_df.csv", index=False)
ts_year.assign(theme=config_name).to_csv(produce_stats_dir / "all_ts_year_df.csv", index=False)
ts_quarter.assign(theme=config_name).to_csv(produce_stats_dir / "all_ts_quarter_df.csv", index=False)

# EXPORT CHART DATA AS CSVs
# Save chart-ready data for Google Sheets visualization
ts_year.to_csv(chart_csvs_dir / f"timeseries_year_chart_data_{config_name}.csv", index=False)
ts_quarter.to_csv(chart_csvs_dir / f"timeseries_quarter_chart_data_{config_name}.csv", index=False)

print("✅ Data processing complete:")
print(f"  - Projects: {len(df)}")
print(f"  - Years covered: {ts_year['year'].min()}-{ts_year['year'].max()}")
print(f"  - Quarters covered: {len(ts_quarter)}")
print(f"  - Consolidated data saved to: {produce_stats_dir}")
print(f"  - Chart data exported to: {chart_csvs_dir}")

print("\n📊 Time Series Summary (with imputation):")
display(ts_year.head(10))
print(f"\nQuarterly data sample (with imputation):")
display(ts_quarter.head(10))

✅ Data processing complete:
  - Projects: 26
  - Years covered: 2014-2025
  - Quarters covered: 48
  - Consolidated data saved to: /Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/data/upfront_costs_innovation/gtr/produce_stats
  - Chart data exported to: /Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/data/upfront_costs_innovation/gtr/produce_stats/chart_csvs

📊 Time Series Summary (with imputation):


/var/folders/0d/x9z6wprs7cd865gzfgfcbxyw0000gp/T/ipykernel_10344/4038984778.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts_quarter_for_imputation['quarter_datetime'] = pd.to_datetime(ts_quarter_for_imputation['quarter'])


,n_projects,amount_millions,year
0,1.0,0.148012,2014
1,0.0,0.000000,2015
2,2.0,0.111151,2016
3,1.0,0.685319,2017
4,0.0,0.000000,2018
5,2.0,5.495649,2019
6,3.0,0.357598,2020
7,3.0,1.747745,2021
8,6.0,0.239605,2022
9,3.0,0.898224,2023



Quarterly data sample (with imputation):


,n_projects,amount_millions,quarter
0,1.0,0.148012,2014Q1
1,0.0,0.000000,2014Q2
2,0.0,0.000000,2014Q3
3,0.0,0.000000,2014Q4
4,0.0,0.000000,2015Q1
5,0.0,0.000000,2015Q2
6,0.0,0.000000,2015Q3
7,0.0,0.000000,2015Q4
8,0.0,0.000000,2016Q1
9,0.0,0.000000,2016Q2


## Analysis and Visualization

In [7]:
# Enhanced Chart Generation with Organized Saving
import altair as alt

# Projects per year chart
chart1 = alt.Chart(ts_year).mark_bar().encode(
    x="year:O",
    y="n_projects:Q",
    tooltip=["n_projects"]
).properties(
    title=f"{config_name.replace('_', ' ').title()} – number of projects per year (GtR)",
    width=600,
    height=300
)

# Save both HTML and PNG versions
chart1.save(str(charts_dir / f"{config_name}_projects_per_year.html"))
chart1.save(str(charts_dir / f"{config_name}_projects_per_year.png"))

print(f"✅ Projects per year chart saved to: {charts_dir}")

✅ Projects per year chart saved to: /Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/data/upfront_costs_innovation/gtr/produce_stats/charts


In [8]:
# Enhanced Chart Generation - Funding per Year
import altair as alt

# Funding per year chart
chart2 = alt.Chart(ts_year).mark_bar().encode(
        x="year:O",
        y=alt.Y("amount_millions:Q", title="Amount (£ millions)"),
        tooltip=["amount_millions"]
    ).properties(
        title=f"{config_name.replace('_', ' ').title()} – funding awarded per year (GtR)",
        width=600,
        height=300
)

# Save chart in multiple formats
chart2.save(str(charts_dir / f"{config_name}_funding_per_year.html"))
chart2.save(str(charts_dir / f"{config_name}_funding_per_year.png"))

# Save chart data for Google Sheets integration
chart_data = ts_year[['year', 'amount_millions']].copy()
chart_data.to_csv(chart_csvs_dir / f"{config_name}_funding_per_year_chart_data.csv", index=False)

In [9]:
# Enhanced Chart Generation - Projects per Quarter
import altair as alt

# Projects per quarter chart
chart3 = alt.Chart(ts_quarter).mark_bar().encode(
    x="quarter:O",
    y="n_projects:Q",
    tooltip=["n_projects"]
).properties(
    title=f"{config_name.replace('_', ' ').title()} – number of projects per quarter (GtR)",
    width=600,
    height=300
)

# Save chart in multiple formats
chart3.save(str(charts_dir / f"{config_name}_projects_per_quarter.html"))
chart3.save(str(charts_dir / f"{config_name}_projects_per_quarter.png"))

# Save chart data for Google Sheets integration
chart_data = ts_quarter[['quarter', 'n_projects']].copy()
chart_data.to_csv(chart_csvs_dir / f"{config_name}_projects_per_quarter_chart_data.csv", index=False)

## 🔗 Google Sheets Integration

Upload the analysis results to Google Sheets for sharing and further analysis. The upload includes both the main dataset and chart data for visualization.

In [11]:
# Google Sheets Upload Configuration
from discovery_utils.utils import google

# Set your Google Sheet IDs here - one for each config
sheet_ids = {
    "upfront_costs": "1M3XgpxaJsAKDzQNbXU6rtBa7ZrNhi6fI5uOPNrqqbBE"  # GTR Upfront Cost Innovation
}

# Optionally specify a single sheet to update (e.g. "llm_filtered_projects"); set to None to update all
SHEET_TO_UPDATE = None  

# Define columns for GtR data upload
cols_projects = [
    "id", "title", "abstractText", "start", "end", "amount", "year", "quarter",
    "theme", "_score_keywords", "_score_vectors", "_score_avg", "ai_relevance_check"
]

# Process and upload MAIN DATA using simplified directory structure
config_name = "upfront_costs"
print(f"Processing {config_name} for Google Sheets upload (MAIN DATA ONLY)...")

# Get the sheet ID for this config
sheet_id = sheet_ids[config_name]

try:
    # Check if we have the required data files from the simplified structure
    if not produce_stats_dir.exists():
        print(f"❌ Analysis not run yet - produce_stats directory not found")
        print("Please run the analysis sections above first.")
    else:
        # Load relevant data from raw and filtered directories
        relevant_df_path = raw_dir / f"relevant_{config_name}.csv"
        relevant_check_path = raw_dir / f"llm_check_MS_{config_name}.jsonl"
        
        if relevant_df_path.exists() and relevant_check_path.exists():
            relevant_df = pd.read_csv(relevant_df_path)
            relevant_check_df = pd.read_json(relevant_check_path, lines=True)
            relevant_checked_df = relevant_df.merge(
                relevant_check_df[['id', 'is_relevant']], on='id', how='left'
            ).rename(columns={"is_relevant": "ai_relevance_check"})
            
            # Add theme column
            relevant_checked_df['theme'] = config_name
            
            # Filter data to match available columns
            available_cols = [col for col in cols_projects if col in relevant_checked_df.columns]
            missing_cols = [col for col in cols_projects if col not in relevant_checked_df.columns]
            
            if missing_cols:
                print(f"  - Warning: Missing columns will be skipped: {missing_cols}")
            
            _all_projects = relevant_checked_df[available_cols]
            
            # ALSO export only the LLM-approved rows
            _llm_only = _all_projects.query("ai_relevance_check == 'yes'")
            
            # Create time series data if available
            try:
                # Try to create basic time series from the filtered data
                if 'start' in _llm_only.columns and 'amount' in _llm_only.columns:
                    # Convert start date and extract year
                    _llm_only_ts = _llm_only.copy()
                    _llm_only_ts['start_date'] = pd.to_datetime(_llm_only_ts['start'])
                    _llm_only_ts['year'] = _llm_only_ts['start_date'].dt.year
                    
                    # Create time series aggregations
                    ts_year = _llm_only_ts.groupby('year').agg({
                        'id': 'count',
                        'amount': 'sum'
                    }).reset_index().rename(columns={
                        'id': 'n_projects',
                        'amount': 'total_amount'
                    })
                    ts_year['theme'] = config_name
                    
                    # Create quarterly data
                    _llm_only_ts['quarter'] = _llm_only_ts['start_date'].dt.to_period('Q').astype(str)
                    ts_quarter = _llm_only_ts.groupby('quarter').agg({
                        'id': 'count',
                        'amount': 'sum'
                    }).reset_index().rename(columns={
                        'id': 'n_projects',
                        'amount': 'total_amount'
                    })
                    ts_quarter['theme'] = config_name
                    
                else:
                    # Create empty time series if columns not available
                    ts_year = pd.DataFrame(columns=['year', 'n_projects', 'total_amount', 'theme'])
                    ts_quarter = pd.DataFrame(columns=['quarter', 'n_projects', 'total_amount', 'theme'])
                    
            except Exception as e:
                print(f"  - Warning: Could not create time series data: {e}")
                ts_year = pd.DataFrame(columns=['year', 'n_projects', 'total_amount', 'theme'])
                ts_quarter = pd.DataFrame(columns=['quarter', 'n_projects', 'total_amount', 'theme'])
            
            print(f"Data prepared for {config_name}:")
            print(f"  - Projects: {len(_all_projects)} (pre-LLM), {len(_llm_only)} (LLM-approved)")
            print(f"  - Time series (yearly): {len(ts_year)} data points")
            print(f"  - Time series (quarterly): {len(ts_quarter)} data points")
            
            # Upload MAIN DATA to Google Sheets with clean sheet names
            sheet_names = {
                "gtr_projects": _all_projects,
                "llm_filtered_projects": _llm_only,
                "gtr_timeseries_year": ts_year,
                "gtr_timeseries_quarter": ts_quarter
            }
            
            # Only upload non-empty dataframes (and optionally limit to one sheet)
            if SHEET_TO_UPDATE:
                upload_data = {name: df for name, df in sheet_names.items()
                               if name == SHEET_TO_UPDATE and not df.empty}
            else:
                upload_data = {name: df for name, df in sheet_names.items() if not df.empty}
            
            print(f"  - Preparing to upload {len(upload_data)} MAIN DATA sheets")
            
            if upload_data:
                google.upload_data_to_gsheet(sheet_id, upload_data)
                print(f"✓ Uploaded {len(upload_data)} MAIN DATA sheets for {config_name} to Google Sheet: {sheet_id}")
                
                # Apply formatting to key sheets with error handling
                try:
                    google.format_gsheet(sheet_id, "gtr_projects", freeze_cols=3)
                    google.format_gsheet(sheet_id, "llm_filtered_projects", freeze_cols=3)
                            
                except Exception as e:
                    print(f"  - Warning: Could not apply formatting: {e}")
                
            else:
                print(f"⚠️  No data to upload for {config_name}")
                
        else:
            print(f"⚠️  Could not find raw data files for {config_name}")
            print("Make sure you've run the data processing sections above first.")
            
except Exception as e:
    print(f"❌ Error uploading {config_name} to Google Sheets: {e}")
    import traceback
    traceback.print_exc()

print("\n🎯 Main data Google Sheets upload complete!")

Processing upfront_costs for Google Sheets upload (MAIN DATA ONLY)...
  - Warning: Missing columns will be skipped: ['year', 'quarter']
Data prepared for upfront_costs:
  - Projects: 366 (pre-LLM), 26 (LLM-approved)
  - Time series (yearly): 11 data points
  - Time series (quarterly): 16 data points
  - Preparing to upload 4 MAIN DATA sheets
2025-07-30 17:25:30,341 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:25:34,237 - root - INFO - Uploading DataFrame to sheet: gtr_projects


/Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-30 17:25:51,603 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:25:54,945 - root - INFO - Uploading DataFrame to sheet: llm_filtered_projects


/Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-30 17:26:10,488 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:26:13,533 - root - INFO - Uploading DataFrame to sheet: gtr_timeseries_year


/Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-30 17:26:28,498 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:26:31,361 - root - INFO - Uploading DataFrame to sheet: gtr_timeseries_quarter


/Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-30 17:26:44,316 - root - INFO - Upload completed successfully.
✓ Uploaded 4 MAIN DATA sheets for upfront_costs to Google Sheet: 1M3XgpxaJsAKDzQNbXU6rtBa7ZrNhi6fI5uOPNrqqbBE
2025-07-30 17:26:45,482 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:26:49,792 - root - INFO - Connected to Google Sheet: LCH innovation

🎯 Main data Google Sheets upload complete!


In [ ]:
# Google Sheets Upload - CHART DATA ONLY
# Run this cell separately to upload just the chart data when needed

# Use the same sheet ID as above for chart data
chart_sheet_id = sheet_ids["upfront_costs"]  # Use the same sheet as main data

# Optionally specify a single chart to update, or None for all charts
CHART_TO_UPDATE = None  # e.g. "chart_timeseries_year" or None for all

print("🔥 Starting CHART DATA upload...")

# Process and upload CHART DATA using simplified directory structure
config_name = "upfront_costs"
print(f"Processing {config_name} for Chart Data upload...")

try:
    chart_data = {}
    
    # Define chart files based on what the notebook generates
    chart_files = {
        "chart_projects_per_year": f"{config_name}_projects_per_year_chart_data.csv",
        "chart_funding_per_year": f"{config_name}_funding_per_year_chart_data.csv",
        "chart_projects_per_quarter": f"{config_name}_projects_per_quarter_chart_data.csv"
    }
    
    # Load chart data with error handling
    for tab_name, filename in chart_files.items():
        chart_file_path = chart_csvs_dir / filename
        if chart_file_path.exists():
            try:
                chart_df = pd.read_csv(chart_file_path)
                if not chart_df.empty:
                    chart_data[tab_name] = chart_df
                    print(f"  ✓ Loaded {tab_name}: {len(chart_df)} rows")
                else:
                    print(f"  ⚠️  Chart file is empty: {filename}")
            except Exception as e:
                print(f"  ❌ Error loading chart file {filename}: {e}")
        else:
            print(f"  ⚠️  Chart file not found: {filename}")
    
    # Filter to specific chart if requested
    if CHART_TO_UPDATE:
        if CHART_TO_UPDATE in chart_data:
            chart_data = {CHART_TO_UPDATE: chart_data[CHART_TO_UPDATE]}
            print(f"  - Uploading only: {CHART_TO_UPDATE}")
        else:
            print(f"  ⚠️  Requested chart {CHART_TO_UPDATE} not found")
    
    print(f"  - Chart datasets ready: {len(chart_data)}")
    
    if chart_data:
        # Upload chart data to Google Sheets
        google.upload_data_to_gsheet(chart_sheet_id, chart_data)
        print(f"✓ Uploaded {len(chart_data)} CHART sheets for {config_name} to Google Sheet: {chart_sheet_id}")
        
        # Apply formatting to chart sheets
        try:
            for chart_tab in chart_data.keys():
                google.format_gsheet(chart_sheet_id, chart_tab, freeze_cols=1)
            print(f"  ✓ Applied formatting to {len(chart_data)} chart sheets")
                    
        except Exception as e:
            print(f"  - Warning: Could not apply formatting to charts: {e}")
        
    else:
        print(f"⚠️  No chart data to upload for {config_name}")
        
except Exception as e:
    print(f"❌ Error uploading chart data for {config_name}: {e}")
    import traceback
    traceback.print_exc()

print("\n🎯 Chart data Google Sheets upload complete!")

## 📊 Analysis Summary & Next Steps

### Key Outputs Generated:
- **Main Dataset**: Filtered upfront cost innovation research projects saved to `filtered/{config_name}_filtered.csv`
- **Time Series Analysis**: Projects and funding trends by year and quarter
- **Visualizations**: Interactive charts saved in HTML and PNG formats
- **Chart Data**: CSV files for Google Sheets visualization integration

### Directory Structure (Simplified):
```
/data/upfront_costs_innovation/gtr/
├── raw/                    # Original data files
├── filtered/               # LLM-filtered results
├── produce_stats/          # Statistical summaries
├── charts/                 # Visualization files (.html, .png)
└── chart_csvs/            # Chart data for Google Sheets
```

### Google Sheets Integration:
- Main data uploaded to dedicated spreadsheet
- Individual chart data available in separate worksheets
- Ready for collaborative analysis and reporting

### Technical Notes:
- Uses OpenAI API for relevance filtering
- Simplified file structure for better data management
- Error handling and data validation throughout
- Compatible with existing pipeline architecture

---

## 🔧 Troubleshooting

**Expected Results**:
- Upfront Cost Innovation Projects: 50-80 research projects
- Research funding focused on academic institutions
- Time series showing UKRI funding patterns over multiple years

**Common Issues**:
1. **Missing Configuration File**:
   ```
   Error: FileNotFoundError: config_upfront_costs.yaml
   Solution: Ensure YAML config file exists in notebook directory
   ```

2. **LLM Processing Timeouts**:
   ```
   Error: LLM batch processing timeout
   Solution: Reduce batch_size from 15 to 10, increase sleep_time to 1.0
   ```

3. **Google Sheets Upload Failures**:
   ```
   Error: 403 Insufficient permissions
   Solution: Verify service account has edit access to target sheet
   ```

4. **Chart Generation Errors**:
   ```
   Error: Chart save failed
   Solution: Ensure charts/ directory exists and has write permissions
   ```

**Data Quality Validation**:
- If results are significantly outside expected ranges, check:
  - Search term configuration in YAML file
  - SCORE_THRESHOLD setting (try 0.2-0.4 range)
  - LLM relevance checking logic

**Performance Tips**:
- **For development**: Set `SHEET_TO_UPDATE` and `CHART_TO_UPDATE` for partial uploads
- **For speed**: Use existing filtered data when possible (don't delete filtered/ dirs)
- **For debugging**: Check intermediate CSV files in raw/ directories
- **For memory**: Restart kernel if processing multiple configs sequentially